# Модуль 3: Retrieval Tuning & Reranking

**Production RAG Pipeline — Часть 3**

В Модулях 1-2 мы построили базовый pipeline: `документ → чанки → embeddings → similarity search`.
Recall@3 = 0.62, Precision@3 = 0.47 — есть куда расти.

В этом модуле:
1. **BM25** — sparse retrieval (классический текстовый поиск)
2. **Hybrid Search** — объединяем dense (embeddings) + sparse (BM25) через RRF
3. **Cross-Encoder Reranking** — переранжируем top-K для точности
4. **Advanced Retrieval** — HyDE, Multi-Query, Parent Document
5. **Evaluation** — метрики качества RAG (Recall, MRR, NDCG)
6. **Full Pipeline** — собираем всё вместе

**Аналогия для Java-разработчика:**
- BM25 ≈ `LIKE '%query%'` в SQL, но умнее (TF-IDF)
- Embeddings ≈ `vector_cosine_similarity()` в pgvector
- Hybrid = оба вместе, как составной индекс
- Reranking ≈ `ORDER BY relevance_score` после начальной выборки

**Пререквизит:** Модуль 2 (Embeddings & Vector Stores)

In [ ]:
# Установка зависимостей (запустить один раз)
!pip install -q rank_bm25 sentence-transformers chromadb langchain-text-splitters tiktoken

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)
import chromadb
import re
import time
import gc
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("Все библиотеки загружены")

## Подготовка данных из Модулей 1-2

Используем тот же юридический документ + добавляем второй для более реалистичного теста.

In [ ]:
LEGAL_DOC_1 = """
# Регламент обработки обращений граждан

## 1. Общие положения

Настоящий регламент устанавливает порядок обработки обращений граждан в электронной форме. Регламент распространяется на все подразделения организации, участвующие в процессе обработки обращений.

Обращение гражданина — это направленное в организацию письменное предложение, заявление или жалоба, а также устное обращение гражданина. Каждое обращение должно быть зарегистрировано в системе в течение одного рабочего дня с момента поступления.

## 2. Классификация обращений

### 2.1 По типу

Обращения классифицируются следующим образом:
- **Предложение** — рекомендация по совершенствованию деятельности организации
- **Заявление** — просьба о содействии в реализации прав и свобод
- **Жалоба** — требование о восстановлении нарушенных прав

### 2.2 По приоритету

Приоритет обращения определяется автоматически на основе следующих критериев:
- **Высокий**: жалобы на нарушение прав, обращения от льготных категорий граждан, повторные обращения
- **Средний**: заявления, требующие межведомственного взаимодействия
- **Низкий**: предложения, информационные запросы

## 3. Процедура обработки

### 3.1 Регистрация

При поступлении обращения оператор выполняет следующие действия:
1. Проверяет полноту заполнения обязательных полей (ФИО, контактные данные, суть обращения)
2. Присваивает регистрационный номер в формате ОГ-ГГГГ-НННННН
3. Определяет тип и приоритет обращения
4. Направляет обращение ответственному исполнителю

### 3.2 Рассмотрение

Ответственный исполнитель обязан:
- Изучить суть обращения в течение 3 рабочих дней
- Запросить дополнительные документы при необходимости
- Подготовить проект ответа
- Согласовать ответ с руководителем подразделения

Срок рассмотрения обращения не должен превышать 30 календарных дней с момента регистрации. В исключительных случаях срок может быть продлён ещё на 30 дней с уведомлением заявителя.

### 3.3 Контроль сроков

Система автоматически отслеживает сроки обработки обращений:
- За 5 дней до истечения срока — уведомление исполнителю
- За 1 день — уведомление исполнителю и руководителю
- При просрочке — эскалация на уровень руководства организации

## 4. Ответственность

За нарушение сроков рассмотрения обращений граждан предусмотрена дисциплинарная ответственность в соответствии с трудовым законодательством Российской Федерации. Повторное нарушение сроков может являться основанием для применения мер дисциплинарного взыскания.
"""

LEGAL_DOC_2 = """
# Положение о персональных данных

## 1. Основные понятия

Персональные данные — любая информация, относящаяся к прямо или косвенно определённому физическому лицу (субъекту персональных данных). К персональным данным относятся: ФИО, дата рождения, адрес, телефон, email, паспортные данные, ИНН, СНИЛС.

Оператор персональных данных — организация, самостоятельно или совместно с другими лицами организующая обработку персональных данных, определяющая цели обработки и состав данных.

## 2. Принципы обработки

### 2.1 Законность и справедливость

Обработка персональных данных осуществляется на законной и справедливой основе. Не допускается обработка данных, несовместимая с целями их сбора. Обработка ограничивается достижением конкретных, заранее определённых и законных целей.

### 2.2 Минимизация данных

Содержание и объём обрабатываемых персональных данных должны соответствовать заявленным целям обработки. Не допускается избыточность данных по отношению к целям их обработки.

## 3. Права субъекта данных

### 3.1 Право на доступ

Субъект персональных данных имеет право получить информацию об обработке его данных, включая:
- Подтверждение факта обработки данных
- Правовые основания и цели обработки
- Способы обработки
- Сроки хранения данных
- Перечень действий с данными

### 3.2 Право на удаление

Субъект данных вправе требовать от оператора уничтожения его персональных данных, если:
- Данные являются незаконно полученными
- Данные не соответствуют заявленным целям обработки
- Истёк срок действия согласия на обработку
- Субъект отзывает согласие на обработку

Оператор обязан уничтожить данные в течение 30 дней с момента получения требования.

## 4. Защита данных

### 4.1 Технические меры

Организация обязана применять следующие технические меры защиты:
- Шифрование данных при передаче (TLS 1.2+) и хранении (AES-256)
- Разграничение доступа на основе ролевой модели (RBAC)
- Журналирование всех операций с персональными данными
- Регулярное резервное копирование
- Проведение аудита безопасности не реже 1 раза в год

### 4.2 Инцидент безопасности

При обнаружении утечки персональных данных оператор обязан:
1. Уведомить Роскомнадзор в течение 24 часов
2. Уведомить субъектов данных в течение 72 часов
3. Провести внутреннее расследование
4. Принять меры по минимизации последствий
5. Задокументировать инцидент и предпринятые меры

## 5. Ответственность

Нарушение законодательства о персональных данных влечёт административную ответственность в виде штрафа: для должностных лиц — до 100 000 рублей, для юридических лиц — до 18 000 000 рублей. За систематические нарушения предусмотрена уголовная ответственность.
"""

# --- Chunking из Модулей 1-2 ---
def chunk_document(doc, chunk_size=400, chunk_overlap=50):
    """Комбинированный chunking: Markdown Headers + Recursive."""
    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Документ"),
            ("##", "Раздел"),
            ("###", "Подраздел"),
        ],
        strip_headers=False,
    )
    header_chunks = md_splitter.split_text(doc)

    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    chunks = []
    for chunk in header_chunks:
        if len(chunk.page_content) > chunk_size:
            sub_chunks = recursive_splitter.create_documents(
                [chunk.page_content], metadatas=[chunk.metadata]
            )
            chunks.extend(sub_chunks)
        else:
            chunks.append(chunk)
    return chunks


chunks_1 = chunk_document(LEGAL_DOC_1)
chunks_2 = chunk_document(LEGAL_DOC_2)
all_chunks = chunks_1 + chunks_2

texts = [c.page_content for c in all_chunks]
metadatas = [c.metadata for c in all_chunks]

print(f"Документ 1 (обращения): {len(chunks_1)} чанков")
print(f"Документ 2 (перс. данные): {len(chunks_2)} чанков")
print(f"Всего: {len(all_chunks)} чанков\n")

for i, c in enumerate(all_chunks):
    doc_name = c.metadata.get("Документ", "?")[:30]
    section = c.metadata.get("Раздел", c.metadata.get("Подраздел", ""))[:30]
    print(f"  {i:2d}. [{len(c.page_content):>3} симв.] {doc_name} / {section}")

---

## Part 1: BM25 — Sparse Retrieval

**BM25** (Best Matching 25) — классический алгоритм текстового поиска. Это улучшенная версия TF-IDF.

Как работает:
- **TF (Term Frequency):** чем чаще слово в чанке → тем релевантнее (с насыщением)
- **IDF (Inverse Document Frequency):** редкие слова важнее частых ("Роскомнадзор" > "является")
- **Длина документа:** короткие чанки с совпадением ранжируются выше

**Плюсы BM25 (vs embeddings):**
- Точное совпадение терминов ("ОГ-ГГГГ-НННННН", "Роскомнадзор", "AES-256")
- Не нужна GPU/модель — работает на CPU мгновенно
- Прозрачный: можно объяснить, почему документ найден

**Минусы:**
- Не понимает синонимы ("жалоба" ≠ "претензия")
- Не понимает смысл — только совпадение слов

**Java-аналогия:** Lucene/Elasticsearch под капотом используют BM25 как дефолтный scoring.

In [ ]:
def tokenize_ru(text):
    """Простая токенизация для русского текста.
    Убираем пунктуацию, приводим к нижнему регистру, убираем стоп-слова.
    """
    # Стоп-слова (частые слова без семантики)
    stop_words = {
        "и", "в", "на", "с", "по", "не", "для", "от", "из", "к", "о", "а",
        "или", "но", "что", "это", "как", "при", "его", "её", "их", "до",
        "все", "так", "за", "же", "то", "ещё", "уже", "бы", "ли", "ни",
        "об", "во", "со", "без", "под", "над", "между", "через", "после",
        "также", "может", "быть", "должен", "должна", "должно", "являться",
    }
    # Оставляем только буквы и цифры
    words = re.findall(r"[а-яёa-z0-9]+", text.lower())
    return [w for w in words if w not in stop_words and len(w) > 1]


# Токенизируем все чанки
tokenized_corpus = [tokenize_ru(text) for text in texts]

# Создаём BM25 индекс
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 индекс: {len(tokenized_corpus)} документов")
print(f"\nПример токенизации чанка 0:")
print(f"  Исходный: {texts[0][:100]}...")
print(f"  Токены:   {tokenized_corpus[0][:15]}...")

In [ ]:
def bm25_search(query, bm25_index, texts, top_k=5):
    """Поиск по BM25."""
    tokenized_query = tokenize_ru(query)
    scores = bm25_index.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(idx, scores[idx]) for idx in top_indices if scores[idx] > 0]


# Тест 1: Точный терминологический запрос (BM25 силён)
query = "Роскомнадзор уведомление 24 часа"
results = bm25_search(query, bm25, texts)

print(f'Запрос: "{query}"')
print(f"Токены запроса: {tokenize_ru(query)}\n")
for rank, (idx, score) in enumerate(results[:3]):
    print(f"  {rank+1}. [score={score:.3f}] Чанк {idx}: {texts[idx][:120]}...\n")

In [ ]:
# Тест 2: Семантический запрос (BM25 слаб)
query = "как защитить личную информацию клиентов"
results = bm25_search(query, bm25, texts)

print(f'Запрос: "{query}"')
print(f"Токены запроса: {tokenize_ru(query)}\n")

if results:
    for rank, (idx, score) in enumerate(results[:3]):
        print(f"  {rank+1}. [score={score:.3f}] Чанк {idx}: {texts[idx][:120]}...\n")
else:
    print("  Ничего не найдено!")

print("BM25 не нашёл 'персональные данные' по запросу 'личная информация' — нет совпадения слов.")
print("Для семантических запросов нужны embeddings.")

---

## Part 2: Hybrid Search — Dense + Sparse

Идея: объединяем лучшее от обоих миров:
- **BM25 (sparse)** — ловит точные термины, аббревиатуры, коды
- **Embeddings (dense)** — ловит смысл, синонимы, перефразирования

### Reciprocal Rank Fusion (RRF)

Самый популярный способ объединения двух ранжирований:

```
RRF_score(doc) = Σ  1 / (k + rank_i(doc))
```

Где `k` — параметр сглаживания (обычно 60), `rank_i` — позиция документа в i-м ранжировании.

**Почему RRF работает:**
- Нормализует скоры из разных источников (BM25 скоры ≠ cosine similarity)
- Документы, найденные обоими методами, получают высший ранг
- Документы, найденные одним методом, тоже попадают (не теряем recall)

**Java-аналогия:** как `UNION ALL` + `ROW_NUMBER()` от двух запросов, потом `ORDER BY combined_rank`.

In [ ]:
# Создаём dense retriever (E5 + ChromaDB из Модуля 2)
model = SentenceTransformer("intfloat/multilingual-e5-base")

embeddings = model.encode(
    ["passage: " + t for t in texts],
    normalize_embeddings=True,
    show_progress_bar=True,
)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("hybrid_docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="hybrid_docs",
    metadata={"hnsw:space": "cosine"},
)
collection.add(
    ids=[f"chunk_{i}" for i in range(len(texts))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "section": m.get("Раздел", ""),
            "subsection": m.get("Подраздел", ""),
            "doc": m.get("Документ", "")[:50],
        }
        for m in metadatas
    ],
)

print(f"Dense индекс: {collection.count()} документов, E5-base (768-dim)")

In [ ]:
def dense_search(query, model, collection, top_k=10):
    """Поиск через embeddings (dense retrieval)."""
    query_emb = model.encode(
        ["query: " + query], normalize_embeddings=True
    )
    results = collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=top_k,
        include=["distances"],
    )
    # Возвращаем [(chunk_idx, distance), ...]
    return [
        (int(doc_id.split("_")[1]), dist)
        for doc_id, dist in zip(results["ids"][0], results["distances"][0])
    ]


def hybrid_search_rrf(query, bm25_index, model, collection, texts, top_k=5, rrf_k=60):
    """Hybrid search: BM25 + Dense с Reciprocal Rank Fusion."""
    # Sparse (BM25)
    bm25_results = bm25_search(query, bm25_index, texts, top_k=top_k * 2)
    
    # Dense (Embeddings)
    dense_results = dense_search(query, model, collection, top_k=top_k * 2)
    
    # RRF fusion
    rrf_scores = {}
    sources = {}  # отслеживаем, откуда пришёл документ
    
    for rank, (idx, score) in enumerate(bm25_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank + 1)
        sources[idx] = sources.get(idx, set())
        sources[idx].add("BM25")
    
    for rank, (idx, dist) in enumerate(dense_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank + 1)
        sources[idx] = sources.get(idx, set())
        sources[idx].add("Dense")
    
    # Сортируем по RRF score
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [(idx, score, sources[idx]) for idx, score in ranked]


# Тест: запрос, где оба метода полезны
query = "уведомление Роскомнадзор при утечке данных"

print(f'Запрос: "{query}"\n')

# Только BM25
bm25_res = bm25_search(query, bm25, texts, top_k=3)
print("BM25 (sparse):")
for rank, (idx, score) in enumerate(bm25_res[:3]):
    print(f"  {rank+1}. Чанк {idx} [score={score:.3f}] {texts[idx][:80]}...")

# Только Dense
dense_res = dense_search(query, model, collection, top_k=3)
print("\nDense (embeddings):")
for rank, (idx, dist) in enumerate(dense_res[:3]):
    print(f"  {rank+1}. Чанк {idx} [dist={dist:.4f}] {texts[idx][:80]}...")

# Hybrid RRF
hybrid_res = hybrid_search_rrf(query, bm25, model, collection, texts, top_k=3)
print("\nHybrid (RRF):")
for rank, (idx, score, src) in enumerate(hybrid_res[:3]):
    src_str = "+".join(sorted(src))
    print(f"  {rank+1}. Чанк {idx} [rrf={score:.4f}] [{src_str}] {texts[idx][:80]}...")

In [ ]:
# Сравнение: где BM25 лучше, где Dense лучше, где Hybrid побеждает

test_queries = [
    # Точные термины → BM25 лучше
    ("формат ОГ-ГГГГ-НННННН регистрационный номер", "точный термин"),
    ("AES-256 шифрование TLS", "технические аббревиатуры"),
    
    # Семантика → Dense лучше
    ("как защитить личную информацию клиентов", "семантический запрос"),
    ("что делать если кто-то украл данные", "разговорный запрос"),
    
    # Смешанный → Hybrid лучше
    ("штраф за нарушение закона о персональных данных", "точный + семантика"),
    ("сроки ответа на жалобу гражданина 30 дней", "факт + контекст"),
]

print(f"{'Запрос':<55} | {'BM25 top-1':>10} | {'Dense top-1':>11} | {'Hybrid top-1':>12} | Тип")
print("-" * 110)

for query, qtype in test_queries:
    bm25_res = bm25_search(query, bm25, texts, top_k=1)
    dense_res = dense_search(query, model, collection, top_k=1)
    hybrid_res = hybrid_search_rrf(query, bm25, model, collection, texts, top_k=1)
    
    bm25_top = f"Чанк {bm25_res[0][0]}" if bm25_res else "—"
    dense_top = f"Чанк {dense_res[0][0]}" if dense_res else "—"
    hybrid_top = f"Чанк {hybrid_res[0][0]}" if hybrid_res else "—"
    
    print(f"{query[:55]:<55} | {bm25_top:>10} | {dense_top:>11} | {hybrid_top:>12} | {qtype}")

---

## Part 3: Cross-Encoder Reranking

### Bi-Encoder vs Cross-Encoder

**Bi-Encoder** (то, что мы использовали):
```
query  → Encoder → vector_q ─┐
                               ├─ cosine_similarity → score
doc    → Encoder → vector_d ─┘
```
- Быстрый: можно закэшировать embeddings документов
- Но query и doc кодируются **независимо** — теряются тонкие взаимосвязи

**Cross-Encoder:**
```
[CLS] query [SEP] document [SEP] → Transformer → score
```
- Query и document обрабатываются **вместе** — модель "видит" оба текста
- Точнее, но медленнее (нельзя кэшировать)

**Стратегия: Retrieve → Rerank**
1. Bi-Encoder (быстрый) → top-20 кандидатов
2. Cross-Encoder (точный) → переранжировать top-20 → top-3 финальных

**Java-аналогия:** как двухэтапная фильтрация:
1. `WHERE` с индексом (быстро, грубо) → 20 строк
2. `ORDER BY complex_score()` (медленно, точно) → top-3

In [ ]:
# Загружаем Cross-Encoder (мультиязычная модель для reranking)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Cross-Encoder загружен: ms-marco-MiniLM-L-6-v2")
print("Обучен на MS MARCO — крупнейший датасет для passage retrieval")
print("Работает с русским текстом (multilingual tokenizer)")

In [ ]:
def retrieve_and_rerank(query, model, collection, cross_encoder, texts, top_k_retrieve=10, top_k_final=3):
    """Двухэтапный retrieval: bi-encoder → cross-encoder reranking."""
    # Stage 1: Retrieve top-K candidates (быстро)
    candidates = dense_search(query, model, collection, top_k=top_k_retrieve)
    candidate_indices = [idx for idx, _ in candidates]
    candidate_texts = [texts[idx] for idx in candidate_indices]
    
    # Stage 2: Rerank with Cross-Encoder (точно)
    pairs = [[query, text] for text in candidate_texts]
    ce_scores = cross_encoder.predict(pairs)
    
    # Сортируем по cross-encoder score
    scored = list(zip(candidate_indices, ce_scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    
    return scored[:top_k_final]


# Демонстрация: как reranking меняет порядок
query = "ответственность за утечку персональных данных штраф"

print(f'Запрос: "{query}"\n')

# Stage 1: Bi-Encoder retrieval
candidates = dense_search(query, model, collection, top_k=8)
print("Stage 1 — Bi-Encoder (top-8 кандидатов):")
for rank, (idx, dist) in enumerate(candidates):
    print(f"  {rank+1}. Чанк {idx} [dist={dist:.4f}] {texts[idx][:90]}...")

# Stage 2: Cross-Encoder reranking
reranked = retrieve_and_rerank(query, model, collection, cross_encoder, texts, top_k_retrieve=8, top_k_final=3)
print("\nStage 2 — Cross-Encoder Reranking (top-3):")
for rank, (idx, ce_score) in enumerate(reranked):
    print(f"  {rank+1}. Чанк {idx} [ce_score={ce_score:.4f}] {texts[idx][:90]}...")

print("\n→ Cross-Encoder может поднять релевантный чанк, который bi-encoder поставил ниже.")

In [ ]:
# Полный pipeline: Hybrid Search + Reranking

def full_pipeline(query, bm25_index, model, collection, cross_encoder, texts,
                  top_k_retrieve=10, top_k_final=3, rrf_k=60):
    """Hybrid Retrieval (BM25 + Dense + RRF) → Cross-Encoder Reranking."""
    # Stage 1: Hybrid retrieval
    hybrid_results = hybrid_search_rrf(
        query, bm25_index, model, collection, texts,
        top_k=top_k_retrieve, rrf_k=rrf_k,
    )
    candidate_indices = [idx for idx, _, _ in hybrid_results]
    candidate_texts = [texts[idx] for idx in candidate_indices]
    
    if not candidate_texts:
        return []
    
    # Stage 2: Cross-Encoder reranking
    pairs = [[query, text] for text in candidate_texts]
    ce_scores = cross_encoder.predict(pairs)
    
    scored = list(zip(candidate_indices, ce_scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    
    return scored[:top_k_final]


# Тест полного pipeline
test_queries = [
    "порядок рассмотрения жалоб граждан",
    "шифрование персональных данных AES",
    "что будет за просрочку ответа на обращение",
    "как удалить мои данные из системы",
    "кто уведомляет при утечке данных",
]

for query in test_queries:
    results = full_pipeline(query, bm25, model, collection, cross_encoder, texts)
    top_chunk = results[0] if results else None
    print(f'\n"{query}"')
    if top_chunk:
        idx, score = top_chunk
        doc = metadatas[idx].get("Документ", "")[:30]
        sec = metadatas[idx].get("Раздел", metadatas[idx].get("Подраздел", ""))[:30]
        print(f"  → Чанк {idx} [{doc} / {sec}] (ce={score:.3f})")
        print(f"    {texts[idx][:120]}...")

---

## Part 4: Advanced Retrieval Strategies

### 4.1 HyDE — Hypothetical Document Embeddings

Проблема: запрос пользователя короткий ("штраф за утечку"), а документы длинные и подробные.
Embedding короткого запроса может быть далёк от embedding подробного документа.

**Идея HyDE:**
1. LLM генерирует **гипотетический ответ** на запрос (без доступа к документам)
2. Этот ответ кодируется в embedding
3. Ищем документы, похожие на **гипотетический ответ** (а не на запрос)

```
Query: "штраф за утечку данных"
  ↓ LLM
HyDE: "За утечку персональных данных предусмотрена административная
       ответственность. Штраф для юридических лиц составляет до 18 млн..."
  ↓ encode()
embedding гипотетического ответа
  ↓ similarity search
Находим РЕАЛЬНЫЕ документы, похожие на гипотетический ответ
```

**Плюс:** embedding длинного текста ближе к embedding документов, чем embedding короткого запроса.

**Минус:** нужен вызов LLM (задержка + стоимость), LLM может нагаллюцинировать.

### 4.2 Multi-Query Retrieval

**Идея:** один запрос → LLM генерирует 3-5 перефразировок → ищем по каждой → объединяем.

```
Query: "штраф за утечку"
  ↓ LLM
Q1: "административная ответственность за нарушение закона о персональных данных"
Q2: "размер штрафа для организации при утечке информации"
Q3: "наказание за разглашение персональных данных"
  ↓ search each
  ↓ union + deduplicate
Расширенный набор релевантных документов
```

**Оба метода требуют LLM API.** Ниже — реализация с заглушкой (без реального API-вызова), чтобы показать механику.

In [ ]:
# HyDE — демонстрация механики (без реального LLM)

def hyde_search(query, hypothetical_answer, model, collection, top_k=5):
    """HyDE: ищем по embedding гипотетического ответа вместо запроса."""
    # Кодируем гипотетический ответ как passage (не query!)
    hyde_emb = model.encode(
        ["passage: " + hypothetical_answer],
        normalize_embeddings=True,
    )
    results = collection.query(
        query_embeddings=hyde_emb.tolist(),
        n_results=top_k,
        include=["distances"],
    )
    return [
        (int(doc_id.split("_")[1]), dist)
        for doc_id, dist in zip(results["ids"][0], results["distances"][0])
    ]


query = "штраф за утечку данных"

# В production это генерирует LLM. Здесь — ручная заглушка:
hypothetical = (
    "За утечку персональных данных предусмотрена административная ответственность. "
    "Штраф для должностных лиц составляет до 100 000 рублей, для юридических лиц — "
    "до 18 000 000 рублей. Оператор обязан уведомить Роскомнадзор в течение 24 часов "
    "и субъектов данных в течение 72 часов."
)

print(f'Запрос: "{query}"')
print(f"\nГипотетический ответ (HyDE):")
print(f"  {hypothetical[:200]}...\n")

# Обычный поиск
normal_res = dense_search(query, model, collection, top_k=3)
print("Обычный Dense Search:")
for rank, (idx, dist) in enumerate(normal_res):
    print(f"  {rank+1}. Чанк {idx} [dist={dist:.4f}] {texts[idx][:80]}...")

# HyDE поиск
hyde_res = hyde_search(query, hypothetical, model, collection, top_k=3)
print("\nHyDE Search:")
for rank, (idx, dist) in enumerate(hyde_res):
    print(f"  {rank+1}. Чанк {idx} [dist={dist:.4f}] {texts[idx][:80]}...")

print("\n→ HyDE часто находит более точные чанки, потому что гипотетический ответ")
print("  семантически ближе к реальному документу, чем короткий запрос.")

In [ ]:
# Multi-Query — демонстрация механики

def multi_query_search(queries, model, collection, top_k_per_query=5, top_k_final=5):
    """Multi-Query: ищем по нескольким перефразировкам, объединяем результаты."""
    all_results = {}  # idx -> min distance
    
    for q in queries:
        results = dense_search(q, model, collection, top_k=top_k_per_query)
        for idx, dist in results:
            if idx not in all_results or dist < all_results[idx]:
                all_results[idx] = dist
    
    # Сортируем по лучшему distance
    ranked = sorted(all_results.items(), key=lambda x: x[1])[:top_k_final]
    return ranked


original_query = "как удалить свои данные"

# В production эти варианты генерирует LLM:
expanded_queries = [
    original_query,
    "право на удаление персональных данных",
    "отзыв согласия на обработку информации",
    "уничтожение данных по требованию субъекта",
]

print(f'Оригинальный запрос: "{original_query}"')
print(f"Расширенные запросы:")
for i, q in enumerate(expanded_queries[1:], 1):
    print(f"  {i}. {q}")

# Обычный поиск
single_res = dense_search(original_query, model, collection, top_k=3)
print("\nОбычный поиск (1 запрос):")
for rank, (idx, dist) in enumerate(single_res):
    print(f"  {rank+1}. Чанк {idx} [dist={dist:.4f}] {texts[idx][:80]}...")

# Multi-Query поиск
multi_res = multi_query_search(expanded_queries, model, collection, top_k_final=3)
print("\nMulti-Query поиск (4 запроса):")
for rank, (idx, dist) in enumerate(multi_res):
    print(f"  {rank+1}. Чанк {idx} [dist={dist:.4f}] {texts[idx][:80]}...")

print("\n→ Multi-Query расширяет охват: находит документы, которые")
print("  не нашлись бы по одной формулировке запроса.")

### 4.3 Parent Document Retriever

Проблема: маленькие чанки → точный поиск, но **мало контекста** для LLM.
Большие чанки → много контекста, но **размытый embedding** → плохой поиск.

**Идея:**
1. Храним документы на **двух уровнях**: маленькие (для поиска) + большие (для LLM)
2. Ищем по маленьким чанкам → возвращаем **родительский** большой чанк

```
Документ
├── Parent chunk (800 символов) — отдаём в LLM
│   ├── Child chunk 1 (200 символов) — ищем по embedding
│   ├── Child chunk 2 (200 символов)
│   └── Child chunk 3 (200 символов)
├── Parent chunk (800 символов)
│   ├── Child chunk 4 (200 символов)
│   └── Child chunk 5 (200 символов)
...
```

**Java-аналогия:** как `@OneToMany` — ищем по дочерним, загружаем родительскую.

In [ ]:
# Parent Document Retriever

# Уровень 1: большие parent-чанки (для контекста LLM)
parent_chunks = chunk_document(LEGAL_DOC_1, chunk_size=800, chunk_overlap=100)
parent_chunks += chunk_document(LEGAL_DOC_2, chunk_size=800, chunk_overlap=100)
parent_texts = [c.page_content for c in parent_chunks]

# Уровень 2: маленькие child-чанки (для точного поиска)
child_to_parent = {}  # child_idx -> parent_idx
child_texts = []
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

for parent_idx, parent in enumerate(parent_texts):
    children = child_splitter.split_text(parent)
    for child in children:
        child_idx = len(child_texts)
        child_texts.append(child)
        child_to_parent[child_idx] = parent_idx

print(f"Parent чанков: {len(parent_texts)} (avg {np.mean([len(t) for t in parent_texts]):.0f} символов)")
print(f"Child чанков:  {len(child_texts)} (avg {np.mean([len(t) for t in child_texts]):.0f} символов)")

# Индексируем child-чанки
child_embeddings = model.encode(
    ["passage: " + t for t in child_texts],
    normalize_embeddings=True,
    show_progress_bar=True,
)

try:
    chroma_client.delete_collection("child_docs")
except Exception:
    pass

child_collection = chroma_client.create_collection(
    name="child_docs",
    metadata={"hnsw:space": "cosine"},
)
child_collection.add(
    ids=[f"child_{i}" for i in range(len(child_texts))],
    documents=child_texts,
    embeddings=child_embeddings.tolist(),
    metadatas=[{"parent_idx": child_to_parent[i]} for i in range(len(child_texts))],
)

print(f"\nChild-индекс: {child_collection.count()} документов")

In [ ]:
def parent_doc_search(query, model, child_collection, parent_texts, child_to_parent, top_k=3):
    """Ищем по child-чанкам, возвращаем parent-чанки."""
    query_emb = model.encode(["query: " + query], normalize_embeddings=True)
    results = child_collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=top_k * 2,  # берём больше, т.к. дедупликация по parent
        include=["documents", "metadatas", "distances"],
    )
    
    seen_parents = set()
    final_results = []
    
    for i in range(len(results["ids"][0])):
        parent_idx = results["metadatas"][0][i]["parent_idx"]
        if parent_idx not in seen_parents:
            seen_parents.add(parent_idx)
            final_results.append({
                "parent_idx": parent_idx,
                "parent_text": parent_texts[parent_idx],
                "child_text": results["documents"][0][i],
                "distance": results["distances"][0][i],
            })
        if len(final_results) >= top_k:
            break
    
    return final_results


query = "уничтожение персональных данных по требованию"
print(f'Запрос: "{query}"\n')

results = parent_doc_search(query, model, child_collection, parent_texts, child_to_parent)

for i, res in enumerate(results):
    print(f"--- Результат {i+1} (distance: {res['distance']:.4f}) ---")
    print(f"Найден child: \"{res['child_text'][:80]}...\"")
    print(f"Возвращён parent ({len(res['parent_text'])} символов):")
    print(f"  {res['parent_text'][:200]}...\n")

print("→ Поиск по маленькому child-чанку, но LLM получает полный parent-контекст.")

---

## Part 5: Evaluation — метрики качества Retrieval

Как понять, что retrieval стал лучше? Нужны метрики.

| Метрика | Что измеряет | Формула |
|---------|-------------|----------|
| **Recall@K** | Доля найденных релевантных из всех релевантных | `\|retrieved ∩ relevant\| / \|relevant\|` |
| **Precision@K** | Доля релевантных среди найденных | `\|retrieved ∩ relevant\| / K` |
| **MRR** (Mean Reciprocal Rank) | Как высоко стоит первый релевантный результат | `1/rank_of_first_relevant` |
| **NDCG@K** | Качество ранжирования с учётом позиций | Логарифмический discount |
| **F1@K** | Баланс Precision и Recall | `2 * P * R / (P + R)` |

**Для RAG самое важное:**
- **Recall@K** — нашли ли мы нужный чанк? (если не нашли → LLM не сможет ответить)
- **MRR** — насколько высоко релевантный чанк? (чем выше → меньше шума для LLM)

In [ ]:
# Ground Truth для evaluation
# Для каждого запроса — список индексов релевантных чанков (ручная разметка)

eval_queries = [
    {
        "query": "порядок рассмотрения жалоб граждан",
        "relevant": [i for i, t in enumerate(texts) if "жалоб" in t.lower() or "рассмотрен" in t.lower()],
    },
    {
        "query": "сроки обработки обращений",
        "relevant": [i for i, t in enumerate(texts) if "срок" in t.lower() and "обращен" in t.lower()],
    },
    {
        "query": "регистрационный номер обращения формат",
        "relevant": [i for i, t in enumerate(texts) if "регистрац" in t.lower() and "номер" in t.lower()],
    },
    {
        "query": "штраф за нарушение закона о персональных данных",
        "relevant": [i for i, t in enumerate(texts) if "штраф" in t.lower() or ("ответственность" in t.lower() and "данн" in t.lower())],
    },
    {
        "query": "шифрование данных при хранении",
        "relevant": [i for i, t in enumerate(texts) if "шифрован" in t.lower() or "aes" in t.lower()],
    },
    {
        "query": "право на удаление персональных данных",
        "relevant": [i for i, t in enumerate(texts) if "удален" in t.lower() or "уничтож" in t.lower()],
    },
    {
        "query": "действия при утечке данных инцидент",
        "relevant": [i for i, t in enumerate(texts) if "утечк" in t.lower() or "инцидент" in t.lower()],
    },
    {
        "query": "эскалация при просрочке обращения",
        "relevant": [i for i, t in enumerate(texts) if "эскалац" in t.lower() or "просрочк" in t.lower()],
    },
]

print(f"Подготовлено {len(eval_queries)} запросов для evaluation:")
for eq in eval_queries:
    print(f"  \"{eq['query']}\" → {len(eq['relevant'])} релевантных чанков (idx: {eq['relevant']})")

In [ ]:
def evaluate_retriever(retriever_fn, eval_queries, top_k=3):
    """Evaluate retriever по Recall@K, Precision@K, MRR, F1@K."""
    recalls = []
    precisions = []
    mrrs = []
    
    for eq in eval_queries:
        query = eq["query"]
        relevant = set(eq["relevant"])
        
        if not relevant:
            continue
        
        # Получаем результаты (список индексов)
        retrieved = retriever_fn(query, top_k)
        retrieved_set = set(retrieved)
        
        # Recall@K
        hits = len(retrieved_set & relevant)
        recall = hits / len(relevant)
        recalls.append(recall)
        
        # Precision@K
        precision = hits / len(retrieved) if retrieved else 0
        precisions.append(precision)
        
        # MRR (Mean Reciprocal Rank)
        rr = 0
        for rank, idx in enumerate(retrieved):
            if idx in relevant:
                rr = 1.0 / (rank + 1)
                break
        mrrs.append(rr)
    
    avg_recall = np.mean(recalls)
    avg_precision = np.mean(precisions)
    avg_mrr = np.mean(mrrs)
    f1 = 2 * avg_precision * avg_recall / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0
    
    return {
        "Recall@K": avg_recall,
        "Precision@K": avg_precision,
        "MRR": avg_mrr,
        "F1@K": f1,
    }


# Определяем retrievers для сравнения
K = 3

def bm25_retriever(query, top_k):
    results = bm25_search(query, bm25, texts, top_k=top_k)
    return [idx for idx, _ in results[:top_k]]

def dense_retriever(query, top_k):
    results = dense_search(query, model, collection, top_k=top_k)
    return [idx for idx, _ in results[:top_k]]

def hybrid_retriever(query, top_k):
    results = hybrid_search_rrf(query, bm25, model, collection, texts, top_k=top_k)
    return [idx for idx, _, _ in results[:top_k]]

def hybrid_rerank_retriever(query, top_k):
    results = full_pipeline(query, bm25, model, collection, cross_encoder, texts, top_k_retrieve=10, top_k_final=top_k)
    return [idx for idx, _ in results[:top_k]]


# Запускаем evaluation
retrievers = {
    "BM25 (sparse)": bm25_retriever,
    "Dense (E5)": dense_retriever,
    "Hybrid (RRF)": hybrid_retriever,
    "Hybrid + Rerank": hybrid_rerank_retriever,
}

print(f"Evaluation: {len(eval_queries)} запросов, top-{K}\n")

header = f"{'Retriever':<20} | {'Recall@3':>10} | {'Precision@3':>12} | {'MRR':>8} | {'F1@3':>8}"
print(header)
print("-" * len(header))

eval_results = {}
for name, fn in retrievers.items():
    metrics = evaluate_retriever(fn, eval_queries, top_k=K)
    eval_results[name] = metrics
    print(
        f"{name:<20} | {metrics['Recall@K']:>10.3f} | {metrics['Precision@K']:>12.3f} | "
        f"{metrics['MRR']:>8.3f} | {metrics['F1@K']:>8.3f}"
    )

In [ ]:
# Визуализация: сравнение retrievers

metric_names = ["Recall@3", "Precision@3", "MRR", "F1@3"]
metric_keys = ["Recall@K", "Precision@K", "MRR", "F1@K"]
retriever_names = list(eval_results.keys())

x = np.arange(len(metric_names))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 6))

colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

for i, (name, metrics) in enumerate(eval_results.items()):
    values = [metrics[k] for k in metric_keys]
    bars = ax.bar(x + i * width, values, width, label=name, color=colors[i], alpha=0.85)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{val:.2f}",
            ha="center", va="bottom", fontsize=8,
        )

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metric_names)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.15)
ax.set_title("Сравнение Retrieval стратегий", fontsize=14)
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Детальный breakdown: какие запросы проблемные?

print("Детальный анализ Hybrid+Rerank (лучший retriever):\n")

for eq in eval_queries:
    query = eq["query"]
    relevant = set(eq["relevant"])
    retrieved = hybrid_rerank_retriever(query, K)
    
    hits = len(set(retrieved) & relevant)
    recall = hits / len(relevant) if relevant else 0
    status = "✓" if recall >= 0.5 else "✗"
    
    print(f"{status} \"{query}\"")
    print(f"  Релевантные: {sorted(relevant)} | Найденные: {retrieved} | Recall: {recall:.2f}")
    if recall < 1.0:
        missed = relevant - set(retrieved)
        if missed:
            print(f"  Пропущены: {sorted(missed)}")
    print()

---

## Part 6: Итоговый Production Pipeline

Собираем финальный pipeline со всеми техниками:

In [ ]:
class ProductionRAGRetriever:
    """Production-grade retriever: Hybrid Search + Cross-Encoder Reranking.
    
    Pipeline:
    1. Ingest: document → markdown chunking → embeddings → ChromaDB + BM25
    2. Search: query → BM25 + Dense → RRF fusion → Cross-Encoder rerank → top-K
    """

    def __init__(
        self,
        bi_encoder_name="intfloat/multilingual-e5-base",
        cross_encoder_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
        chunk_size=400,
        chunk_overlap=50,
    ):
        self.bi_encoder = SentenceTransformer(bi_encoder_name)
        self.cross_encoder = CrossEncoder(cross_encoder_name)
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.chroma_client = chromadb.Client()
        self.collection = None
        self.bm25 = None
        self.texts = []
        self.metadatas = []
        self.chunks = []

    def ingest(self, documents, collection_name="prod_rag"):
        """Загрузить документы в индекс."""
        # Chunking
        self.chunks = []
        for doc in documents:
            self.chunks.extend(chunk_document(doc, self.chunk_size, self.chunk_overlap))

        self.texts = [c.page_content for c in self.chunks]
        self.metadatas = [c.metadata for c in self.chunks]

        # BM25 index
        tokenized = [tokenize_ru(t) for t in self.texts]
        self.bm25 = BM25Okapi(tokenized)

        # Dense index
        embeddings = self.bi_encoder.encode(
            ["passage: " + t for t in self.texts],
            normalize_embeddings=True,
            show_progress_bar=True,
        )

        try:
            self.chroma_client.delete_collection(collection_name)
        except Exception:
            pass

        self.collection = self.chroma_client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )
        self.collection.add(
            ids=[f"chunk_{i}" for i in range(len(self.texts))],
            documents=self.texts,
            embeddings=embeddings.tolist(),
            metadatas=[
                {
                    "section": m.get("Раздел", ""),
                    "subsection": m.get("Подраздел", ""),
                    "doc": m.get("Документ", "")[:50],
                }
                for m in self.metadatas
            ],
        )

        print(f"Загружено {len(self.texts)} чанков из {len(documents)} документов")
        return self

    def search(self, query, top_k=3, top_k_retrieve=10, rrf_k=60, use_reranking=True):
        """Hybrid Search + Reranking."""
        # Stage 1: BM25
        bm25_scores = self.bm25.get_scores(tokenize_ru(query))
        bm25_top = np.argsort(bm25_scores)[::-1][:top_k_retrieve]
        bm25_results = [(idx, bm25_scores[idx]) for idx in bm25_top if bm25_scores[idx] > 0]

        # Stage 2: Dense
        query_emb = self.bi_encoder.encode(["query: " + query], normalize_embeddings=True)
        dense_results_raw = self.collection.query(
            query_embeddings=query_emb.tolist(),
            n_results=top_k_retrieve,
            include=["distances"],
        )
        dense_results = [
            (int(doc_id.split("_")[1]), dist)
            for doc_id, dist in zip(dense_results_raw["ids"][0], dense_results_raw["distances"][0])
        ]

        # Stage 3: RRF Fusion
        rrf_scores = {}
        for rank, (idx, _) in enumerate(bm25_results):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank + 1)
        for rank, (idx, _) in enumerate(dense_results):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank + 1)

        candidates = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k_retrieve]
        candidate_indices = [idx for idx, _ in candidates]

        if not candidate_indices:
            return []

        # Stage 4: Cross-Encoder Reranking
        if use_reranking:
            pairs = [[query, self.texts[idx]] for idx in candidate_indices]
            ce_scores = self.cross_encoder.predict(pairs)
            scored = sorted(zip(candidate_indices, ce_scores), key=lambda x: x[1], reverse=True)
        else:
            scored = [(idx, rrf_scores[idx]) for idx in candidate_indices]

        results = []
        for idx, score in scored[:top_k]:
            results.append({
                "chunk_idx": idx,
                "text": self.texts[idx],
                "metadata": self.metadatas[idx],
                "score": float(score),
            })
        return results


# Создаём и тестируем
rag = ProductionRAGRetriever()
rag.ingest([LEGAL_DOC_1, LEGAL_DOC_2])

print("\n" + "=" * 70)
print("Тест Production Pipeline:")
print("=" * 70)

test_queries = [
    "как зарегистрировать жалобу на нарушение прав",
    "что делать при обнаружении утечки данных",
    "размер штрафа за нарушение закона о ПДн",
    "формат регистрационного номера обращения",
    "RBAC шифрование TLS сертификат",
]

for q in test_queries:
    results = rag.search(q, top_k=2)
    print(f'\n"{q}"')
    for r in results:
        doc = r["metadata"].get("Документ", "")[:25]
        sec = r["metadata"].get("Раздел", r["metadata"].get("Подраздел", ""))[:25]
        print(f"  → [{doc} / {sec}] (score={r['score']:.3f}) {r['text'][:80]}...")

---

## Выводы

| Техника | Когда использовать | Сложность |
|---------|-------------------|----------|
| **BM25** | Точные термины, аббревиатуры, коды | Низкая |
| **Dense (Embeddings)** | Семантический поиск, синонимы | Средняя |
| **Hybrid (RRF)** | Всегда в production — лучше чем каждый по отдельности | Средняя |
| **Cross-Encoder Reranking** | Когда нужна высокая precision | Средняя |
| **HyDE** | Короткие запросы, FAQ | Высокая (нужен LLM) |
| **Multi-Query** | Неоднозначные запросы | Высокая (нужен LLM) |
| **Parent Document** | Когда LLM нужен широкий контекст | Средняя |

### Production Checklist

- [x] Chunking: Markdown Headers + Recursive (Модуль 1)
- [x] Embeddings: multilingual-e5-base (Модуль 2)
- [x] Vector Store: ChromaDB / FAISS (Модуль 2)
- [x] BM25: sparse retrieval для точных терминов (Модуль 3)
- [x] Hybrid: BM25 + Dense + RRF (Модуль 3)
- [x] Reranking: Cross-Encoder (Модуль 3)
- [x] Evaluation: Recall@K, Precision@K, MRR (Модуль 3)
- [ ] LLM Generation: подача контекста в LLM (Модуль 4)
- [ ] End-to-End Evaluation: RAGAS (Модуль 4)

### Что дальше (Модуль 4)

- **Generation:** подключаем LLM для генерации ответов на основе найденных чанков
- **Prompt Engineering:** системные промпты для RAG, chain-of-thought
- **RAGAS:** автоматическая оценка качества (faithfulness, relevancy, answer correctness)
- **End-to-End Pipeline:** вопрос → retrieval → rerank → LLM → ответ